In [ ]:
This notebook checks the XLSX archive contents and validates the sheet XML files so we can diagnose the empty sheet export.</VSCode.Cell>
<VSCode.Cell language="python">from pathlib import Path
import zipfile
from xml.etree import ElementTree as ET

path = Path('data/032026_Sorted PSOC PSIC.xlsx')
print('Workbook path exists:', path.exists())
print('Size:', path.stat().st_size if path.exists() else 'n/a')

with zipfile.ZipFile(path) as book:
    names = sorted(book.namelist())
    print('Archive entries:')
    for name in names[:40]:
        print(' ', name)
    print('... total', len(names), 'entries')

    def xml_text(name):
        try:
            return book.read(name).decode('utf-8')
        except Exception as exc:
            return f'<failed to decode {exc}>'

    if 'xl/workbook.xml' in names:
        root = ET.fromstring(xml_text('xl/workbook.xml'))
        print('\nWorkbook sheets:')
        for sheet in root.findall('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}sheets/{http://schemas.openxmlformats.org/spreadsheetml/2006/main}sheet'):
            print(' -', sheet.attrib.get('name'), sheet.attrib)

    if 'xl/_rels/workbook.xml.rels' in names:
        root = ET.fromstring(xml_text('xl/_rels/workbook.xml.rels'))
        print('\nWorkbook relationships:')
        for rel in root.findall('{http://schemas.openxmlformats.org/package/2006/relationships}Relationship'):
            print(' -', rel.attrib)
</VSCode.Cell>
<VSCode.Cell language="python">from zipfile import ZipFile
from xml.etree import ElementTree as ET

path = Path('data/032026_Sorted PSOC PSIC.xlsx')
with ZipFile(path) as book:
    strings = []
    if 'xl/sharedStrings.xml' in book.namelist():
        root = ET.fromstring(book.read('xl/sharedStrings.xml'))
        strings = [ ''.join(item.itertext()) for item in root.findall('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}si') ]
    print('Shared strings count:', len(strings))
    for i, s in enumerate(strings[:20]):
        print(i, repr(s))

    workbook = ET.fromstring(book.read('xl/workbook.xml'))
    rels = ET.fromstring(book.read('xl/_rels/workbook.xml.rels'))
    rel_map = {rel.attrib['Id']: rel.attrib['Target'] for rel in rels}
    print('\nSheet targets:')
    for sheet in workbook.findall('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}sheets/{http://schemas.openxmlformats.org/spreadsheetml/2006/main}sheet'):
        rid = sheet.attrib.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
        target = rel_map.get(rid)
        print(' -', sheet.attrib.get('name'), '->', target)

    if 'xl/worksheets/sheet1.xml' in book.namelist():
        data = ET.fromstring(book.read('xl/worksheets/sheet1.xml'))
        rows = data.findall('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}sheetData/{http://schemas.openxmlformats.org/spreadsheetml/2006/main}row')
        print('Sheet1 rows count:', len(rows))
        for row in rows[:5]:
            print(' row', row.attrib.get('r'), [cell.attrib.get('r') for cell in row.findall('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}c')])
